# Training and evaluation

Splits the features by date, trains LightGBM and evaluates it.
Features come from `build_features()` in `src/sentinel/features/pipeline.py`; the split from `split_by_time()` in `src/sentinel/model/split.py`.

In [1]:
from pathlib import Path

import pandas as pd
pd.set_option("display.float_format", "{:,.4f}".format)

from sentinel.features.pipeline import FEATURE_COLUMNS, build_features

RAW_DIR = Path("..") / "data" / "raw"

train_raw = pd.read_csv(RAW_DIR / "fraudTrain.csv", index_col=0)
test_raw = pd.read_csv(RAW_DIR / "fraudTest.csv", index_col=0)
df = pd.concat([train_raw, test_raw], ignore_index=True)

features = build_features(df)
print("features:", features.shape)

features: (1852394, 31)


## Step 7a: time-based split

Splits the features into four date ranges with `split_by_time()`.

**Checks:** the parts add up to every row (no overlaps, no gaps), each part covers the expected dates, and every part contains fraud.

In [2]:
from sentinel.model.split import split_by_time

parts = split_by_time(features)

summary = pd.DataFrame({
    name: {
        "rows": len(part),
        "first": part["trans_date_trans_time"].min(),
        "last": part["trans_date_trans_time"].max(),
        "frauds": part["is_fraud"].sum(),
        "fraud_rate_%": part["is_fraud"].mean() * 100,
    }
    for name, part in parts.items()
}).T

print(summary)
print()
print("rows in parts:", sum(len(p) for p in parts.values()), "| total:", len(features))

            rows                first                 last frauds fraud_rate_%
train     924850  2019-01-01 00:00:18  2019-12-31 23:59:53   5220       0.5644
validate  172843  2020-01-01 00:00:57  2020-03-31 23:59:55   1123       0.6497
test      229040  2020-04-01 00:03:39  2020-06-30 23:59:50   1296       0.5658
replay    525661  2020-07-01 00:00:08  2020-12-31 23:59:34   2012       0.3828

rows in parts: 1852394 | total: 1852394


## Step 7b: baselines

The simplest possible scores on the **validation** set, without any training. LightGBM must beat these to justify its complexity.

| Baseline | Score used |
|---|---|
| random | A random number per transaction (the floor) |
| amount only | The raw `amt` |
| amount ratio only | `amt_ratio_30d`, this amount ÷ the customer's 30-day average |

In [3]:
import numpy as np
from sklearn.metrics import average_precision_score

val = parts["validate"]
y_val = val["is_fraud"]

rng = np.random.default_rng(42)

baselines = {
    "random": rng.random(len(val)),
    "amount only": val["amt"],
    "amount ratio only": val["amt_ratio_30d"].fillna(0),
}

print(f"fraud rate in validation: {y_val.mean():.4f}")
print()
for name, score in baselines.items():
    print(f"{name:<20} PR-AUC = {average_precision_score(y_val, score):.4f}")

fraud rate in validation: 0.0065

random               PR-AUC = 0.0066
amount only          PR-AUC = 0.2121
amount ratio only    PR-AUC = 0.1052


## Step 7c: training LightGBM

Trains `train_model()` from `src/sentinel/model/train.py` twice: without and with fraud weighting (`scale_pos_weight`).
Early stopping picks the number of trees using validation PR-AUC.

**Bar to beat:** amount-only baseline, PR-AUC 0.2121.

In [4]:
from sentinel.model.train import train_model

train, validate = parts["train"], parts["validate"]

models = {}
for weighted in (False, True):
    model = train_model(train, validate, weight_fraud=weighted)
    scores = model.predict_proba(validate[FEATURE_COLUMNS])[:, 1]
    pr_auc = average_precision_score(validate["is_fraud"], scores)
    models[weighted] = model
    print(f"weighted={weighted!s:<5}  trees={model.best_iteration_:>4}  validation PR-AUC={pr_auc:.4f}")

weighted=False  trees= 682  validation PR-AUC=0.9253
weighted=True   trees= 746  validation PR-AUC=0.9209


## Step 7d: does the score depend on the fraud-only cards?

91 cards in Sparkov contain only fraud (found in exploration). Their transactions have almost no history, which could teach the model a "little history = fraud" shortcut.

Compares validation PR-AUC **with** and **without** those cards' transactions.

In [5]:
per_card = features.groupby("cc_num").size()
fraud_only_cards = per_card[per_card < 100].index

model = models[False]
val_scores = pd.Series(model.predict_proba(validate[FEATURE_COLUMNS])[:, 1], index=validate.index)

is_fraud_only = validate["cc_num"].isin(fraud_only_cards)
normal = ~is_fraud_only

print("validation frauds from fraud-only cards:", validate.loc[is_fraud_only, "is_fraud"].sum(), "of", validate["is_fraud"].sum())
print()
print(f"PR-AUC, all transactions:        {average_precision_score(validate['is_fraud'], val_scores):.4f}")
print(f"PR-AUC, without fraud-only cards: {average_precision_score(validate.loc[normal, 'is_fraud'], val_scores[normal]):.4f}")

validation frauds from fraud-only cards: 76 of 1123

PR-AUC, all transactions:        0.9253
PR-AUC, without fraud-only cards: 0.9196
